In [9]:
!pip install lyricsgenius

In [10]:
pip install pandas lyricsgenius nltk better-profanity langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993222 sha256=c00ade909278d6a7a0c25f940fe74ae692a14eb544f67ab4526702cda5455795
  Stored in directory: /Users/Geva/Library/Caches/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
Note: you may need to restart the kernel to use updated packages.


In [13]:
import random
import re
import lyricsgenius
import pandas as pd

token = "MM3rhwjFkBZVZU5k7fixqT4u_rCu9u7kPnLCEWZdKiEoxm0ggcRLFvIwbgg98cEL"
genius = lyricsgenius.Genius(token, timeout=30)

genres = ['pop', 'rock', 'rap', 'r-b', 'country', 'indie', 'blues', 'metal', 'folk']
songs_per_genre = 20

def get_song_metadata(title: str, artist_name: str) -> dict:
    search_term = f"{title} {artist_name}".strip()
    try:
        res = genius.search_songs(search_term, per_page=1, page=1)
        hits = res.get('hits', [])
        if hits:
            return hits[0].get('result', {})
    except Exception as e:
        print(f"Metadata search failed for '{title}' by {artist_name}: {e}")
    return {}

all_songs = []

for genre in genres:
    print(f"מושך שירים מז'אנר: {genre}")
    page = 1
    candidate_hits = []

    while page and len(candidate_hits) < songs_per_genre * 2:
        try:
            res = genius.tag(genre, page=page)
        except Exception as e:
            print(f"Failed to fetch tag page for {genre}: {e}")
            break

        if not res or 'hits' not in res:
            break
        candidate_hits.extend(res['hits'])
        page = res.get('next_page')

    if not candidate_hits:
        print(f"No hits found for {genre}.")
        continue

    unique_hits = {hit['url']: hit for hit in candidate_hits}
    selected_hits = random.sample(
        list(unique_hits.values()),
        min(songs_per_genre, len(unique_hits))
    )

    for hit in selected_hits:
        artist_name = hit['artists'][0] if hit['artists'] else ""
        title = hit['title']

        song_info = get_song_metadata(title, artist_name)
        release_year = None
        if song_info.get('release_date_components'):
            release_year = song_info['release_date_components'].get('year')

        lyrics = ""
        try:
            lyrics = genius.lyrics(song_url=hit['url']) or ""
        except Exception as e:
            print(f"Lyrics failed for '{title}': {e}")

        all_songs.append({
            'song_title': title,
            'artist_name': artist_name,
            'release_year': int(release_year) if release_year else None,
            'genre': genre,
            'lyrics': lyrics,
            'pageviews': song_info.get('stats', {}).get('pageviews', 0),
            'featured_artists_count': len(hit['featured_artists']),
            'primary_tag': genre,
            'language': song_info.get('language'),
            'source_url': hit['url']
        })

raw_df = pd.DataFrame(all_songs)
raw_df.to_csv('raw_genius_songs.csv', index=False, encoding='utf-8-sig')
print("הסתיים שלב המשיכה! הקובץ הגולמי נשמר בהצלחה.")

מושך שירים מז'אנר: pop
מושך שירים מז'אנר: rock
מושך שירים מז'אנר: rap
מושך שירים מז'אנר: r-b
מושך שירים מז'אנר: country
מושך שירים מז'אנר: indie
מושך שירים מז'אנר: blues
מושך שירים מז'אנר: metal
מושך שירים מז'אנר: folk
הסתיים שלב המשיכה! הקובץ הגולמי נשמר בהצלחה.


In [16]:
import re
from collections import Counter
import pandas as pd
import nltk
from nltk.corpus import stopwords
from better_profanity import profanity

# --- תיקון השגיאה והורדת משאבי ה-NLP ---
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)  # <-- השורה שפתרה את ה-LookupError

STOP_WORDS = set(stopwords.words('english'))

def analyze_lyrics_advanced(lyrics):
    if not lyrics or not isinstance(lyrics, str):
        return {k: 0 for k in ['total_word_count', 'count_chorus', 'count_verse', 'count_intro', 'count_bridge', 'unique_special_word', 'special_words_ratio', 'special_word_without_chorus', 'unique_ratio_no_repeated_chorus', 'profanity_count', 'profanity_ratio', 'adjectives_count', 'adjectives_ratio']}
    
    # 1. ספירת חלקי השיר
    raw_tags = re.findall(r'\[(.*?)\]', lyrics)
    chorus_count = sum(1 for t in raw_tags if 'chorus' in t.lower())
    verse_count = sum(1 for t in raw_tags if 'verse' in t.lower())
    intro_count = sum(1 for t in raw_tags if 'intro' in t.lower())
    bridge_count = sum(1 for t in raw_tags if 'bridge' in t.lower())
    
    # 2. הסרת פזמונים חוזרים
    blocks = re.split(r'(\[.*?\])', lyrics)
    modified_blocks = []
    chorus_counter = 0
    skip_current_block = False
    
    for block in blocks:
        if block.startswith('[') and block.endswith(']'):
            if 'chorus' in block.lower():
                chorus_counter += 1
                if chorus_counter > 1:
                    skip_current_block = True
                    continue
            skip_current_block = False
            modified_blocks.append(block)
        else:
            if skip_current_block:
                continue
            modified_blocks.append(block)
            
    lyrics_no_repeated_chorus = "".join(modified_blocks)
    
    def get_clean_words(text):
        text = re.sub(r'\[.*?\]', '', text)
        text = re.sub(r'[^\w\s]', '', text).lower()
        return text.split()
    
    words_all = get_clean_words(lyrics)
    words_no_repeated_chorus = get_clean_words(lyrics_no_repeated_chorus)
    
    total_words_count = len(words_all)
    if total_words_count == 0:
        return {k: 0 for k in ['total_word_count', 'count_chorus', 'count_verse', 'count_intro', 'count_bridge', 'unique_special_word', 'special_words_ratio', 'special_word_without_chorus', 'unique_ratio_no_repeated_chorus', 'profanity_count', 'profanity_ratio', 'adjectives_count', 'adjectives_ratio']}
        
    special_words = [w for w in words_all if w not in STOP_WORDS]
    special_words_no_chorus = [w for w in words_no_repeated_chorus if w not in STOP_WORDS]
    
    # 3. ספירת קללות ומילות תואר
    profanity_count = sum(1 for w in special_words if profanity.contains_profanity(w))
    
    tokens = nltk.word_tokenize(re.sub(r'\[.*?\]', '', lyrics))
    tagged_words = nltk.pos_tag(tokens)
    adjectives_count = sum(1 for word, tag in tagged_words if tag in ('JJ', 'JJR', 'JJS'))
    
    # 4. חישוב מדדים ויחסים
    unique_special_word = len(set(special_words))
    total_special_words = len(special_words)
    special_word_without_chorus = len(special_words_no_chorus)
    
    special_words_ratio = unique_special_word / total_special_words if total_special_words > 0 else 0
    unique_ratio_no_repeated_chorus = unique_special_word / special_word_without_chorus if special_word_without_chorus > 0 else 0
    profanity_ratio = profanity_count / total_words_count
    adjectives_ratio = adjectives_count / total_words_count
    
    return {
        'total_word_count': total_words_count,
        'count_chorus': chorus_count,
        'count_verse': verse_count,
        'count_intro': intro_count,
        'count_bridge': bridge_count,
        'unique_special_word': unique_special_word,
        'special_words_ratio': special_words_ratio,
        'special_word_without_chorus': special_word_without_chorus,
        'unique_ratio_no_repeated_chorus': unique_ratio_no_repeated_chorus,
        'profanity_count': profanity_count,
        'profanity_ratio': profanity_ratio,
        'adjectives_count': adjectives_count,
        'adjectives_ratio': adjectives_ratio
    }

# --- טעינת הקובץ הגולמי ועיבודו ---
print("טוען קובץ גולמי ומעבד נתונים...")
df = pd.read_csv('raw_genius_songs.csv')

# הרצת הניתוח על כל שורה בנפרד והפיכת התוצאה לעמודות חדשות
nlp_results = df['lyrics'].apply(analyze_lyrics_advanced)
nlp_df = pd.DataFrame(list(nlp_results))

# איחוד הטבלה המקורית עם העמודות החדשות
final_df = pd.concat([df, nlp_df], axis=1)

# שמירת הדאטה-סט הסופי והמועשר
final_df.to_csv('final_genius_enriched_songs.csv', index=False, encoding='utf-8-sig')

print("הסתיים שלב ה-NLP! הקובץ הסופי מוכן:")
print(final_df.info())

טוען קובץ גולמי ומעבד נתונים...
הסתיים שלב ה-NLP! הקובץ הסופי מוכן:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 23 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   song_title                       180 non-null    object 
 1   artist_name                      180 non-null    object 
 2   release_year                     180 non-null    int64  
 3   genre                            180 non-null    object 
 4   lyrics                           180 non-null    object 
 5   pageviews                        180 non-null    int64  
 6   featured_artists_count           180 non-null    int64  
 7   primary_tag                      180 non-null    object 
 8   language                         0 non-null      float64
 9   source_url                       180 non-null    object 
 10  total_word_count                 180 non-null    int64  
 11  count_chorus    

In [17]:
final_df

,song_title,artist_name,release_year,genre,lyrics,pageviews,featured_artists_count,primary_tag,language,source_url,...,count_intro,count_bridge,unique_special_word,special_words_ratio,special_word_without_chorus,unique_ratio_no_repeated_chorus,profanity_count,profanity_ratio,adjectives_count,adjectives_ratio
0,Houdini,Eminem,2024,pop,"[Skit: Paul Rosenberg]\nHey, Em, it's Paul\nUh...",6062903,0,pop,NaN,https://genius.com/Eminem-houdini-lyrics,...,1,1,271,0.624424,401,0.675810,26,0.034946,53,0.071237
1,Love Yourself,Justin Bieber,2015,pop,[Verse 1]\nFor all the times that you rained o...,8235345,0,pop,NaN,https://genius.com/Justin-bieber-love-yourself...,...,0,0,83,0.406863,120,0.691667,0,0.000000,19,0.042986
2,FRIENDS,Marshmello,2018,pop,"[Intro]\nOoh-oh, ooh-woo\nOoh-oh, ooh-woo\n\n[...",5985328,0,pop,NaN,https://genius.com/Marshmello-and-anne-marie-f...,...,1,1,76,0.312757,106,0.716981,3,0.006849,38,0.086758
3,Hotline Bling,Drake,2015,pop,[Intro]\nYou used to call me on my\nYou used t...,7320132,0,pop,NaN,https://genius.com/Drake-hotline-bling-lyrics,...,1,1,75,0.350467,139,0.539568,0,0.000000,4,0.009281
4,The Hills,The Weeknd,2015,pop,[Intro]\nYeah\nYeah\nYeah\n\n[Verse 1]\nYour m...,11041226,0,pop,NaN,https://genius.com/The-weeknd-the-hills-lyrics,...,1,1,86,0.385650,147,0.585034,15,0.032609,34,0.073913
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,Piano Man,Billy Joel,1973,folk,[Verse 1]\nIt's nine o'clock on a Saturday\nTh...,1815999,0,folk,NaN,https://genius.com/Billy-joel-piano-man-lyrics,...,0,0,127,0.638191,163,0.779141,1,0.002786,27,0.075209
176,"Take Me Home, Country Roads",John Denver,1971,folk,"[Verse 1]\nAlmost Heaven, West Virginia\nBlue ...",2299530,0,folk,NaN,https://genius.com/John-denver-take-me-home-co...,...,0,1,55,0.474138,74,0.743243,0,0.000000,4,0.022727
177,Eldest Daughter,Taylor Swift,2025,folk,[Verse 1]\nEverybody's so punk on the internet...,2503212,0,folk,NaN,https://genius.com/Taylor-swift-eldest-daughte...,...,0,1,113,0.545894,106,1.066038,3,0.008451,33,0.092958
178,Fade into You,Mazzy Star,1993,folk,[Verse 1]\nI wanna hold the hand inside you\nI...,2538350,0,folk,NaN,https://genius.com/Mazzy-star-fade-into-you-ly...,...,0,0,44,0.578947,58,0.758621,0,0.000000,6,0.041958
